# Ampliar y limpiar una foto: difusión frente a métodos clásicos

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial — IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento Digital de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Comparar un modelo de difusión con métodos clásicos al ampliar una foto pequeña y al quitarle ruido, y medir la diferencia.

## Al terminar este cuaderno vas a poder:

1. Ampliar una foto 4 veces con un modelo de difusión y con interpolación bicúbica.
2. Quitar ruido con filtros clásicos y con `img2img`, ajustando `strength`.
3. Leer PSNR y SSIM sin confundir "número más alto" con "foto mejor".

## Preparación

Necesita **GPU** (T4). Usamos una foto de Perm (1910, dominio público) de la que **tenemos el original nítido**. Eso nos permite degradarla a propósito y comparar cada restauración con la verdad. En una foto vieja real, esa verdad no existe.

`scikit-image` (métricas) y `opencv` (filtros) ya vienen instalados en Colab.

In [ ]:
%pip install -q diffusers transformers accelerate

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/015_modelos_difusion'  # carpeta de la unidad en tu Drive
DATOS = f'{CARPETA}/datos'  # imágenes de ejemplo

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Sin GPU. Menú: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU')

print('GPU:', torch.cuda.get_device_name(0))
print('Memoria de la GPU:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

In [ ]:
import os

RESULTADOS = f'{CARPETA}/resultados'  # acá guardamos lo que generamos
os.makedirs(RESULTADOS, exist_ok=True)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

completa = Image.open(f'{DATOS}/original_perm_1910.jpg').convert('RGB')
referencia = completa.crop((920, 380, 1432, 892))  # recorte de 512 x 512 con ventanas y molduras
referencia.save(f'{RESULTADOS}/perm_referencia.png')
referencia

## 1. Ampliar: de 128 a 512 píxeles

Simulamos una foto chica: achicamos el recorte a 128 × 128. Después intentamos recuperar los 512 × 512 de dos maneras:

- **Bicúbica**: calcula cada píxel nuevo promediando sus vecinos. No agrega información.
- **Upscaler de difusión** (`stable-diffusion-x4-upscaler`): **genera** detalle plausible.

In [ ]:
baja = referencia.resize((128, 128), Image.LANCZOS)  # la "foto chica"
bicubica = baja.resize((512, 512), Image.BICUBIC)  # método clásico
baja

Parámetros del upscaler:

- `prompt`: describe la foto; el modelo lo usa como pista. No es opcional.
- `num_inference_steps=20`: pasos de limpieza (el valor por defecto es 75, más lento).
- `noise_level=20`: ruido que se le suma a la foto chica antes de ampliarla. Más ruido, más libertad para inventar.
- `enable_attention_slicing()`: calcula la atención por partes y ahorra memoria.

In [ ]:
from diffusers import StableDiffusionUpscalePipeline

pipe_ampliar = StableDiffusionUpscalePipeline.from_pretrained(
    'stabilityai/stable-diffusion-x4-upscaler',
    torch_dtype=torch.float16,
).to('cuda')
pipe_ampliar.enable_attention_slicing()

generador = torch.Generator('cuda').manual_seed(0)
ampliada = pipe_ampliar(
    prompt='a photograph of a historic building, sharp details',
    image=baja,
    num_inference_steps=20,
    noise_level=20,
    generator=generador,
).images[0]

bicubica.save(f'{RESULTADOS}/perm_bicubica.png')
ampliada.save(f'{RESULTADOS}/perm_difusion_x4.png')
print('Tamaño del resultado:', ampliada.size)

> Si la imagen sale negra o con manchas, la causa suele ser el cálculo en `float16`. Cambiá `torch.float16` por `torch.float32` en la carga.

Miramos las tres versiones y un **detalle ampliado** (un recorte de 128 × 128 de cada una), que es donde se notan las diferencias.

In [ ]:
versiones = [('original', referencia), ('bicúbica', bicubica), ('difusión x4', ampliada)]

figura, ejes = plt.subplots(2, 3, figsize=(15, 10))
for numero in range(len(versiones)):
    nombre, imagen = versiones[numero]
    ejes[0, numero].imshow(imagen)
    ejes[0, numero].set_title(nombre)
    ejes[1, numero].imshow(imagen.crop((100, 120, 228, 248)), interpolation='nearest')  # detalle de las ventanas
    ejes[1, numero].set_title(f'{nombre} (detalle)')
    ejes[0, numero].axis('off')
    ejes[1, numero].axis('off')
plt.tight_layout()
plt.show()

## 2. Medir: PSNR y SSIM

Dos números para comparar cada versión con el original:

- **PSNR** (decibeles): qué tan cerca están los píxeles. Más alto, más parecido numérico.
- **SSIM** (0 a 1): qué tan parecida es la estructura. Más cerca de 1, más parecida.

Ambos miden **parecido con el original**, no belleza ni nitidez percibida.

In [ ]:
ref = np.array(referencia)

for nombre, imagen in [('bicúbica', bicubica), ('difusión x4', ampliada)]:
    arreglo = np.array(imagen)
    print(f'{nombre:<12} PSNR {psnr(ref, arreglo):5.2f} dB   SSIM {ssim(ref, arreglo, channel_axis=2):.3f}')

> **Para mirar.** Es común que la bicúbica tenga **mejor** número y la difusión **se vea** más nítida. La difusión agrega detalle que no está en el original; por eso se aleja de él, aunque parezca mejor. Ese detalle es una **suposición** del modelo.

## 3. Limpiar: quitar ruido

Le agregamos ruido gaussiano al recorte (`sigma = 25`; la semilla fija el ruido) y lo restauramos con tres métodos:

- **Mediana** (`cv2.medianBlur`): reemplaza cada píxel por la mediana de su vecindario. Quita puntos aislados.
- **Bilateral** (`cv2.bilateralFilter`): promedia, pero respeta los bordes.
- **Difusión** (`img2img`): ensucia un poco más y deja que el modelo limpie. Acá se usa `strength`.

In [ ]:
del pipe_ampliar  # liberamos la GPU del upscaler
import gc
gc.collect()
torch.cuda.empty_cache()

generador_ruido = np.random.default_rng(0)
ruido = generador_ruido.normal(0, 25, size=ref.shape)  # ruido gaussiano, sigma = 25
con_ruido_arreglo = np.clip(ref + ruido, 0, 255).astype(np.uint8)
con_ruido = Image.fromarray(con_ruido_arreglo)

mediana = cv2.medianBlur(con_ruido_arreglo, 5)  # vecindario de 5 x 5
bilateral = cv2.bilateralFilter(con_ruido_arreglo, d=9, sigmaColor=75, sigmaSpace=75)

con_ruido.save(f'{RESULTADOS}/perm_con_ruido.png')
con_ruido

Parámetros de `img2img`:

- `image`: la foto con ruido.
- `strength`: de 0 a 1, cuánto se ensucia antes de limpiar (cuaderno 01). Probamos tres valores.
- `guidance_scale=5.0`: cuánto empuja el texto. En Stable Diffusion 1.5 se usa de 5 a 8.
- `num_inference_steps=30`: pasos totales; con `strength=0.4` se usan 12 (40 % de 30).

In [ ]:
from diffusers import AutoPipelineForImage2Image

pipe_limpiar = AutoPipelineForImage2Image.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')

fuerzas = [0.2, 0.4, 0.6]
limpias = []
for fuerza in fuerzas:
    generador = torch.Generator('cuda').manual_seed(0)
    resultado = pipe_limpiar(
        prompt='a sharp photograph of a historic building',
        image=con_ruido,
        strength=fuerza,
        guidance_scale=5.0,
        num_inference_steps=30,
        generator=generador,
    ).images[0]
    limpias.append(resultado)
    resultado.save(f'{RESULTADOS}/perm_img2img_{int(fuerza * 100)}.png')

del pipe_limpiar
gc.collect()
torch.cuda.empty_cache()

In [ ]:
resultados = [('con ruido', con_ruido), ('mediana', Image.fromarray(mediana)), ('bilateral', Image.fromarray(bilateral))]
for numero in range(len(fuerzas)):
    resultados.append((f'difusión s={fuerzas[numero]}', limpias[numero]))

figura, ejes = plt.subplots(2, 3, figsize=(15, 10))
for numero in range(len(resultados)):
    nombre, imagen = resultados[numero]
    ejes[numero // 3, numero % 3].imshow(imagen.crop((100, 120, 356, 376)), interpolation='nearest')  # detalle de 256 px
    ejes[numero // 3, numero % 3].set_title(nombre)
    ejes[numero // 3, numero % 3].axis('off')
plt.tight_layout()
plt.show()

for nombre, imagen in resultados:
    arreglo = np.array(imagen)
    print(f'{nombre:<18} PSNR {psnr(ref, arreglo):5.2f} dB   SSIM {ssim(ref, arreglo, channel_axis=2):.3f}')

> **Para mirar.** Con `strength` bajo la difusión casi no cambia la foto; con `strength` alto aparecen texturas nuevas (ladrillos, molduras) que **no estaban**. Los filtros clásicos nunca inventan, pero tampoco recuperan detalle.

> **Para probar.** Repetí con `sigma = 10` y con `sigma = 50`. ¿A partir de qué ruido deja de convenir el filtro clásico?

## Glosario

| Término | Qué significa acá |
|---|---|
| **Interpolación bicúbica** | Ampliar promediando vecinos; no agrega información nueva. |
| **Super-resolución** | Ampliar una imagen recuperando (o generando) detalle. |
| **`img2img`** | Restaurar partiendo de una foto en lugar de ruido puro. |
| **`strength`** | Cuánto se ensucia la foto antes de limpiarla. |
| **PSNR** | Qué tan cerca están los píxeles del original (decibeles). |
| **SSIM** | Qué tan parecida es la estructura al original (0 a 1). |
| **Mediana / bilateral** | Filtros clásicos para quitar ruido. |